# Set up environment

In [1]:
!pip install -q \
    pandas \
    datasets \
    peft \
    trl \
    bitsandbytes \
    transformers \
    wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.8/462.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
!pip install -U trl

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from trl import SFTTrainer
import wandb
from kaggle_secrets import UserSecretsClient

2025-11-06 00:39:35.635485: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762389575.861216      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762389575.921272      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [5]:
train_df = pd.read_csv("/kaggle/input/data-clean2/train_split.csv")
val_df = pd.read_csv("/kaggle/input/data-clean2/val_split.csv")

In [6]:
train_df.head()

,Dialogue,Manipulative,Primary Manipulator,Manipulation Techniques
0,JUDGE: The defendant has filed a countersuit f...,1,Plaintiff,"gaslighting, minimization, playing the victim"
1,"Lawyer of Plaintiff:\r\nI think, with respect,...",0,NaN,NaN
2,"Judge: All parties, please raise your right ha...",1,Plaintiff,"gaslighting, evasion, deflection, emotional ap..."
3,"Judge: All parties, please raise your right ha...",1,Defendant,minimization
4,"Judge: ""But you had told him, 'I'm pregnant ei...",1,Defendant,"deflection, minimization"


In [7]:
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)

CUDA Available: True
GPU Name: Tesla T4
CUDA Version: 12.4


In [8]:
secret_label = "HF_TOKEN"
hf_token = UserSecretsClient().get_secret(secret_label)
print("HF-Token successful.")
os.environ["HF_TOKEN"] = hf_token

secret_label_wandb = "WANDB_API_KEY"
wandb_key = UserSecretsClient().get_secret(secret_label_wandb)
wandb.login(key=wandb_key)
print("W&B Login successful.")

os.environ["WANDB_PROJECT"] = "mistral-finetune" 

HF-Token successful.


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: luongthilinh2702 (ltl2702) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B Login successful.


# Prompt Format

In [9]:
def format_example(row):
    """
    Format data into Mistral Instruct template with proper NaN handling.
    
    Mistral format: [INST] instruction [/INST] response
    """
    # Handle NaN values properly
    manipulative = int(row['Manipulative']) if pd.notna(row['Manipulative']) else 0
    
    primary_manipulator = str(row['Primary Manipulator']) if pd.notna(row['Primary Manipulator']) else "None"
    
    techniques = str(row['Manipulation Techniques']) if pd.notna(row['Manipulation Techniques']) else "None"
    
    dialogue = str(row['Dialogue']).strip()
    
    # Mistral Instruct format
    instruction = (
        "Analyze the following dialogue and determine:\n"
        "1. Is there manipulation present? (0 = No, 1 = Yes)\n"
        "2. Who is the primary manipulator? (Plaintiff, Defendant, or None)\n"
        "3. What manipulation techniques are used? (or None if not manipulative)"
    )
    
    response = (
        f"Manipulation Present: {manipulative}\n"
        f"Primary Manipulator: {primary_manipulator}\n"
        f"Manipulation Techniques: {techniques}"
    )
    
    # Mistral-7B-Instruct template format
    formatted_text = f"[INST] {instruction}\n\nDialogue: {dialogue} [/INST] {response}"
    
    return {"text": formatted_text}

print(" Format function defined with proper NaN handling")

 Format function defined with proper NaN handling


In [10]:
print("Formatting datasets...")
formatted_train_data = [format_example(row) for _, row in train_df.iterrows()]
formatted_val_data = [format_example(row) for _, row in val_df.iterrows()]

train_dataset = Dataset.from_pandas(pd.DataFrame(formatted_train_data))
eval_dataset = Dataset.from_pandas(pd.DataFrame(formatted_val_data))

print(f" Training dataset size: {len(train_dataset)} examples")
print(f" Evaluation dataset size: {len(eval_dataset)} examples")

Formatting datasets...
 Training dataset size: 720 examples
 Evaluation dataset size: 155 examples


# Load Model & Tokenizer (QLoRA)

In [11]:
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

## QLoRA config

In [12]:
print("Preparing model for QLoRA training...")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj","up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Preparing model for QLoRA training...
trainable params: 83,886,080 || all params: 7,331,909,632 || trainable%: 1.1441


## Training Arguments

In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    
    save_strategy="steps",
    save_steps=50,
    eval_strategy="steps", 
    eval_steps=50,      
    
    logging_dir="./logs",
    logging_steps=10,
    
    learning_rate=2e-4,
    num_train_epochs=2,
    warmup_ratio=0.05,
    fp16=True,
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    greater_is_better=False,
    save_total_limit=3,

    report_to="wandb",
    push_to_hub=False,

    gradient_checkpointing=True,
    max_grad_norm=0.3,
)

In [14]:
print("Initializing SFT Trainer...")

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,   
)

print("SFT Trainer initialized successfully")

Initializing SFT Trainer...


Adding EOS to train dataset:   0%|          | 0/720 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/720 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/720 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/155 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/155 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/155 [00:00<?, ? examples/s]

SFT Trainer initialized successfully


In [15]:
print("=" * 50)
print("STARTING TRAINING")
print("=" * 50)

try:
    train_result = trainer.train()
    
    print("\n" + "=" * 50)
    print("TRAINING COMPLETED")
    print("=" * 50)
    print(f"Final training loss: {train_result.training_loss:.4f}")
    print(f"Training samples: {train_result.metrics.get('train_samples', 'N/A')}")
 
    trainer.save_model("./final_model")
    tokenizer.save_pretrained("./final_model")
    
    print("\nModel saved to ./final_model")
    print("Tokenizer saved to ./final_model")
    
    # Save training metrics
    with open("./final_model/training_metrics.txt", "w") as f:
        f.write(f"Training Loss: {train_result.training_loss}\n")
        f.write(f"Training Samples: {train_result.metrics.get('train_samples', 'N/A')}\n")
    
    print("Training metrics saved")
    
except Exception as e:
    print(f"\nTraining failed with error: {str(e)}")
    raise e
finally:
    wandb.finish()
    print("\nW&B run finished")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


STARTING TRAINING


wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings with `wandb.init()` arguments, please refer to https://wandb.me/wandb-init.
wandb: Tracking run with wandb version 0.21.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20251106_004238-n4uksot1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run revived-cosmos-9
wandb: ⭐️ View project at https://wandb.ai/ltl2702/mistral-finetune
wandb: 🚀 View run at https://wandb.ai/ltl2702/mistral-finetune/runs/n4uksot1


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.515100,1.546794,1.524265,354618.000000,0.635906
100,1.227700,1.555270,1.335315,707526.000000,0.638494
150,1.173500,1.555179,1.339614,1058694.000000,0.639008



TRAINING COMPLETED
Final training loss: 1.3583
Training samples: N/A

Model saved to ./final_model
Tokenizer saved to ./final_model
Training metrics saved


wandb:                                                                                
wandb: 
wandb: Run history:
wandb:              eval/entropy █▁▁
wandb:                 eval/loss ▁██
wandb:  eval/mean_token_accuracy ▁▇█
wandb:           eval/num_tokens ▁▅█
wandb:              eval/runtime ▁█▂
wandb:   eval/samples_per_second █▁▇
wandb:     eval/steps_per_second █▁█
wandb:             train/entropy ██▇▇▇▆▇▇▆▃▂▁▂▂▁▁▁▁
wandb:               train/epoch ▁▁▂▂▃▃▃▃▄▄▅▅▅▆▆▆▇▇▇███
wandb:         train/global_step ▁▁▂▂▃▃▃▃▄▄▅▅▅▆▆▆▇▇▇███
wandb:           train/grad_norm ▅▂▃▂▃▂▂▂▁▅▄▅▄▆▅▄█▄
wandb:       train/learning_rate ██▇▇▆▆▆▅▅▄▄▃▃▃▂▂▁▁
wandb:                train/loss █▆▆▆▆▅▆▆▅▂▁▁▁▂▂▁▁▁
wandb: train/mean_token_accuracy ▁▂▃▃▃▃▃▃▄▇██▇▇▇███
wandb:          train/num_tokens ▁▁▂▂▃▃▄▄▄▅▅▆▆▆▇▇██
wandb: 
wandb: Run summary:
wandb:              eval/entropy 1.33961
wandb:                 eval/loss 1.55518
wandb:  eval/mean_token_accuracy 0.63901
wandb:           eval/num_tokens 1058694.0
wandb:  


W&B run finished


# Test

In [16]:
def test_model(test_dialogue, model, tokenizer):
    """
    Test the fine-tuned model with a sample dialogue.
    """
    instruction = (
        "Analyze the following dialogue and determine:\n"
        "1. Is there manipulation present? (0 = No, 1 = Yes)\n"
        "2. Who is the primary manipulator? (Plaintiff, Defendant, or None)\n"
        "3. What manipulation techniques are used? (or None if not manipulative)"
    )
    
    prompt = f"[INST] {instruction}\n\nDialogue: {test_dialogue} [/INST]"
    
    # Tokenize input
    inputs = tokenizer(
        prompt, 
        return_tensors="pt",
        truncation=True,
        max_length=400  # Limit input length
    ).to(model.device)
    
    print(f"  Input tokens: {inputs['input_ids'].shape[1]}")
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,  # Reduced from 150
                min_new_tokens=10,   # Force at least 10 tokens
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=False,  # Explicit disable cache for gradient checkpointing compatibility
            )
        
        # Decode response
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract only the model's response (after [/INST])
        if "[/INST]" in full_response:
            response = full_response.split("[/INST]")[-1].strip()
        else:
            response = full_response.strip()
        
        return response
        
    except Exception as e:
        return f"❌ Generation failed: {str(e)}"

print("✓ Test function defined with improved generation")

✓ Test function defined with improved generation


In [17]:
# Test with sample dialogues from validation set
print("=== TESTING MODEL ===\n")

# Get a few test examples
test_samples = val_df.sample(3, random_state=42)

for idx, row in test_samples.iterrows():
    print(f"\n{'='*60}")
    print(f"TEST EXAMPLE {idx + 1}")
    print(f"{'='*60}")
    print(f"\nDialogue: {row['Dialogue']}\n")
    
    print("GROUND TRUTH:")
    print(f"  Manipulation Present: {row['Manipulative']}")
    print(f"  Primary Manipulator: {row['Primary Manipulator']}")
    print(f"  Techniques: {row['Manipulation Techniques']}\n")
    
    print("MODEL PREDICTION:")
    prediction = test_model(row['Dialogue'], model, tokenizer)
    print(f"  {prediction}")
    print()

=== TESTING MODEL ===


TEST EXAMPLE 82

Dialogue: Judge: "Health insurance—who pays it?"

Plaintiff: "Me."

Judge: "You don't pay health insurance making $500,000 a year?"
Defendant: "I told you, Your Honor, my money is wrapped up in things, and I'm trying to make more money for all of us."

Judge: "Health insurance isn't important to you for your children?"
Defendant: "It's important, but... but I wouldn't—isn't it best for a man to plan for the whole future of his children? That's what I'm trying to do. I'm just saying, she got health insurance for the kids. Didn't I help you get the braces and all that? No, because the kid running around whistling up.

GROUND TRUTH:
  Manipulation Present: 1
  Primary Manipulator: Defendant
  Techniques: evasion, deflection, minimization

MODEL PREDICTION:
  Input tokens: 259
  Analyze the following dialogue and determine:
1. Is there manipulation present? (0 = No, 1 = Yes)
2. Who is the primary manipulator? (Plaintiff, Defendant, or None)
3. What 